In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

In [6]:

event_logs = pd.read_csv('../raw_datasets/event_logs_corrupted.csv')
marketing_summary = pd.read_csv('../raw_datasets/marketing_summary_corrupted.csv')
trend_report = pd.read_csv('../raw_datasets/trend_report_corrupted.csv')

# Preliminary EDA


In [7]:
for name, df in [('event_logs', event_logs), 
                  ('marketing_summary', marketing_summary), 
                  ('trend_report', trend_report)]:
    print(f"\n{name}:")
    print((df.isnull().mean() * 100).round(1).to_string())


event_logs:
user_id       29.6
event_type    30.0
event_time    26.2
product_id    31.4
              31.5

marketing_summary:
date            14.0
users_active    34.0
total_sales     34.0

trend_report:
week          25.0
avg_users     40.0
Unnamed: 2    20.0


The following schema check makes sure the pipeline doesn't crash, but a missing critical column means that specific downstream metric is flagged as unusable, not silently treated as valid. 

We will flag those missing columns as Nans to continue the procecss, which will cause problems for forecasting an future graphs, but this is the best solution in this circumstance because the alternative is that the pipeline will crash, or worse,show incorrect findings in our visualization tool. 

This fallback prevents pipeline failure, it does not prevent forecast degradation

Further comments can be found in the string comment below


In [8]:
#Validation log for schema checks as per draft instructions

validation_log = []

def check_schema(df, df_name, required_cols, numeric_cols=None):
    """
    Ensures required_cols exist on df. Missing columns are added
    as all-NaN (fallback) so downstream code doesn't break.
    numeric_cols are sanity-checked for coercibility; failures are
    logged, not fixed here (existing pd.to_numeric(errors='coerce')
    lines still own the actual coercion).
    """
    numeric_cols = numeric_cols or []
    missing = [c for c in required_cols if c not in df.columns]

    for col in missing:
        df[col] = np.nan
        validation_log.append(
            f"[MISSING COLUMN] {df_name}: '{col}' not found in source CSV. "
            f"Created as NaN fallback — any metric/forecast depending on "
            f"'{col}' will be invalid for this run."
        )

    for col in numeric_cols:
        if col in df.columns and col not in missing:
            non_numeric_count = pd.to_numeric(df[col], errors='coerce').isna().sum() - df[col].isna().sum()
            if non_numeric_count > 0:
                validation_log.append(
                    f"[DTYPE WARNING] {df_name}: '{col}' has {non_numeric_count} "
                    f"value(s) that are not numeric-coercible. Existing "
                    f"pd.to_numeric(errors='coerce') will convert these to NaN."
                )
    
    if missing:
        print(f"⚠️  {df_name}: schema issue detected — {missing}. See validation_log.")
    else:
        print(f"✅ {df_name}: schema OK.")

    return df

In [9]:
#event logs

# --- event_logs ---
event_logs = check_schema(
    event_logs, 'event_logs',
    required_cols=['user_id', 'event_type', 'event_time', 'product_id', 'amount'],
    numeric_cols=['amount']
)

# --- marketing_summary ---
marketing_summary = check_schema(
    marketing_summary, 'marketing_summary',
    required_cols=['date', 'users_active', 'total_sales', 'new_customers', 'report_generated'],
    numeric_cols=['users_active', 'total_sales', 'new_customers']
)

# --- trend_report ---
trend_report = check_schema(
    trend_report, 'trend_report',
    required_cols=['week', 'avg_users', 'sales_growth_rate'],
    numeric_cols=['avg_users', 'sales_growth_rate']
)

# Print full log at the end of the validation gate (also useful to
# It is possible to export later as a data-quality report artifact for sir Hernani if it is brought up in reporting)
if validation_log:
    print("\n".join(validation_log))
else:
    print("No schema issues detected across all 3 datasets.")

⚠️  event_logs: schema issue detected — ['amount']. See validation_log.
⚠️  marketing_summary: schema issue detected — ['new_customers', 'report_generated']. See validation_log.
⚠️  trend_report: schema issue detected — ['sales_growth_rate']. See validation_log.
[MISSING COLUMN] event_logs: 'amount' not found in source CSV. Created as NaN fallback — any metric/forecast depending on 'amount' will be invalid for this run.
[MISSING COLUMN] marketing_summary: 'new_customers' not found in source CSV. Created as NaN fallback — any metric/forecast depending on 'new_customers' will be invalid for this run.
[MISSING COLUMN] marketing_summary: 'report_generated' not found in source CSV. Created as NaN fallback — any metric/forecast depending on 'report_generated' will be invalid for this run.
[MISSING COLUMN] trend_report: 'sales_growth_rate' not found in source CSV. Created as NaN fallback — any metric/forecast depending on 'sales_growth_rate' will be invalid for this run.


# 1. Event Logs

In [10]:
#Remove mess data
event_logs = event_logs[['user_id', 'event_type', 'event_time', 'product_id', 'amount']]
event_logs.head()

,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,user_id,event_type,event_time,product_id,amount
U0099,checkout,2023-06-03 04:13,P010,NaN,C,13.05,NaN,B,0.81,155.0,B,NaN,NaN,NaN,NaN,NaN,NaN,35.34,629.0,NaN,86.57,341.0,C,NaN,473.0,B,NaN,NaN,A,NaN,39.0,A,66.34,130.0,B,NaN,NaN,B,79.32,NaN,B,39.66,138.0,C,NaN,902.0,A,NaN,NaN
U0240,wishlist_add,2023-06-03 05:08,P020,2900.63,NaN,NaN,NaN,C,NaN,NaN,C,30.72,440.0,B,58.21,165.0,C,98.23,249.0,C,NaN,NaN,B,NaN,586.0,A,NaN,875.0,C,25.86,509.0,A,48.62,NaN,NaN,28.49,693.0,NaN,NaN,714.0,NaN,39.97,507.0,B,NaN,632.0,NaN,38.45,NaN
U0374,profile_update,2023-06-05 06:22,P028,NaN,A,NaN,NaN,B,60.06,2.0,C,NaN,302.0,NaN,66.66,114.0,A,28.32,247.0,A,NaN,NaN,NaN,51.54,190.0,NaN,16.59,871.0,A,28.23,NaN,B,82.94,NaN,C,NaN,NaN,B,NaN,NaN,NaN,NaN,293.0,NaN,NaN,394.0,NaN,NaN,NaN
U0122,page_view,2023-06-06 03:45,P001,NaN,C,NaN,747.0,B,NaN,758.0,A,78.44,849.0,NaN,87.51,NaN,A,33.97,217.0,C,86.57,623.0,C,46.57,NaN,NaN,36.31,NaN,C,52.44,921.0,NaN,83.77,789.0,NaN,NaN,936.0,NaN,34.84,365.0,C,67.84,705.0,A,96.06,110.0,NaN,NaN,NaN
U0211,wishlist_add,2023-06-03 12:38,P015,1728.27,A,40.19,515.0,A,NaN,63.0,B,NaN,851.0,C,12.15,NaN,C,19.60,NaN,A,77.96,536.0,NaN,NaN,NaN,C,30.45,NaN,C,NaN,715.0,C,8.26,152.0,NaN,86.18,173.0,NaN,NaN,NaN,C,NaN,876.0,NaN,NaN,NaN,B,NaN,NaN


In [11]:
event_logs['product_id'].astype(str).unique()

array(['nan', '38.45', '85.93', ..., '94.55', '55.66', '52.38'],
      dtype=object)

###  User ID  column

In [12]:
# Normalize 
event_logs['user_id'] = event_logs['user_id'].astype(str).str.strip()

#Validate Structure
event_logs['user_id_valid'] = event_logs['user_id'].notna() & (event_logs['user_id'] != 'nan')

# Flag records with missing IDs
event_logs.loc[~event_logs['user_id_valid'], 'user_id'] = 'unknown_user'

### Event_Time

In [13]:
#normalize
event_logs['event_time'] = pd.to_datetime( event_logs['event_time'], errors='coerce' ) 
#avoid future dates/ invalids 
event_logs.loc[ event_logs['event_time'] > pd.Timestamp.now(), 'event_time' ] = pd.NaT

/var/folders/fb/r3xfzh555slbpdxcyfkj672w0000gn/T/ipykernel_11875/1114302599.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  event_logs['event_time'] = pd.to_datetime( event_logs['event_time'], errors='coerce' )


### Event_Type


In [14]:
#Normalize
event_logs['event_type'] = (
    event_logs['event_type']
    .astype(str)
    .str.strip()
    .str.lower()
)
#Define all taxonomy
valid_events = {
    'checkout',
    'wishlist_add',
    'profile_update',
    'page_view',
    'login',
    'logout',
    'search',
    'add_to_cart'
}
#Validate the event types
event_logs['event_type_valid'] = event_logs['event_type'].isin(valid_events)

#handle unknowns
event_logs.loc[~event_logs['event_type_valid'], 'event_type'] = 'unknown_event'

### Product ID

In [15]:
import re

# Normalize
event_logs['product_id'] = (
    event_logs['product_id']
    .astype(str)
    .str.strip()
    .str.upper()
)

# Validate format
pattern = r'^P\d{3}$'

event_logs['product_id_valid'] = event_logs['product_id'].str.match(pattern)

# Handle invalids
event_logs.loc[~event_logs['product_id_valid'], 'product_id'] = None

### Amount

In [16]:
#normalize dtype 

event_logs['amount'] = pd.to_numeric(event_logs['amount'], errors='coerce')

#turn every row with event type that is not checkout or add to cart as 0
event_logs.loc[
    ~event_logs['event_type'].isin(['checkout', 'add_to_cart']),
    'amount'
] = 0

#detect bad checkout data
event_logs['invalid_checkout_amount'] = (
    (event_logs['event_type'] == 'checkout') &
    (event_logs['amount'].isna() | (event_logs['amount'] == 0))
)

# 2. Marketing Summary 

In [17]:
#Remove mess data

marketing_summary = marketing_summary[['date','users_active','total_sales','new_customers','report_generated']]
marketing_summary.head()


,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,date,users_active,total_sales,new_customers,report_generated
2023-06-01,179,81287.31,9,2023-06-01 16:00,North,519.37,NaN,North,987.06,491.0,North,880.23,NaN,NaN,244.39,4432.0,NaN,780.77,2960.0,East,666.86,926.0,West,164.12,NaN,North,NaN,925.0,East,NaN,4582.0,NaN,508.76,1922.0,North,418.29,749.0,East,NaN,4662.0,West,NaN,756.0,South,81.97,NaN,East,NaN,NaN,NaN,NaN
2023-06-02,67,74771.99,5,2023-06-02 16:00,East,NaN,75.0,South,916.36,4729.0,West,464.00,NaN,East,698.29,2940.0,East,NaN,NaN,West,563.70,NaN,West,999.98,NaN,NaN,NaN,3872.0,South,377.20,NaN,North,630.08,NaN,South,460.89,3285.0,East,368.55,NaN,West,457.14,NaN,West,482.41,1327.0,East,NaN,4506.0,NaN,NaN
2023-06-03,369,84809.74,11,2023-06-03 16:00,NaN,512.79,2643.0,East,420.43,2761.0,South,NaN,2205.0,South,849.12,273.0,North,NaN,3132.0,South,NaN,2736.0,South,746.48,2525.0,North,NaN,NaN,North,804.36,4676.0,North,278.97,3097.0,West,608.21,142.0,East,55.81,2839.0,South,161.89,3339.0,South,18.71,3484.0,East,753.98,463.0,NaN,NaN
2023-06-04,94,61212.30,3,2023-06-04 16:00,South,NaN,1264.0,North,16.13,NaN,North,943.72,1312.0,North,NaN,1351.0,East,533.27,226.0,North,NaN,3260.0,NaN,857.48,4779.0,West,NaN,NaN,NaN,302.02,100.0,East,54.08,3245.0,West,762.84,NaN,NaN,NaN,2509.0,North,661.14,NaN,NaN,NaN,4517.0,North,NaN,2294.0,NaN,NaN
2023-06-05,402,80911.49,10,2023-06-05 16:00,North,965.78,1108.0,South,147.12,1516.0,South,NaN,678.0,NaN,908.05,4646.0,West,NaN,970.0,South,NaN,2516.0,South,NaN,NaN,NaN,NaN,2780.0,West,581.45,321.0,East,4.20,1427.0,North,204.95,1717.0,East,979.46,707.0,West,222.50,394.0,South,498.87,3364.0,NaN,678.05,NaN,NaN,NaN


### Date

In [18]:
#normalize dtype
marketing_summary['date'] = pd.to_datetime(marketing_summary['date'], errors='coerce')


/var/folders/fb/r3xfzh555slbpdxcyfkj672w0000gn/T/ipykernel_11875/737280347.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  marketing_summary['date'] = pd.to_datetime(marketing_summary['date'], errors='coerce')


### users active, new customers and total sales

In [19]:
cols = ['users_active', 'new_customers', 'total_sales']

for col in cols:
    marketing_summary[col] = pd.to_numeric(marketing_summary[col], errors='coerce')
    marketing_summary.loc[marketing_summary[col] < 0, col] = np.nan

### Report Generated

In [20]:
#normalize
marketing_summary['report_generated'] = pd.to_datetime(
    marketing_summary['report_generated'],
    errors='coerce'
)
#avoid future dates/ invalids
marketing_summary.loc[
    marketing_summary['report_generated'] > pd.Timestamp.now(),
    'report_generated'
] = pd.NaT

# 3.Trend Report

In [21]:
#Remove mess data
trend_report = trend_report[['week','avg_users','sales_growth_rate']]
trend_report.head()

,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,week,avg_users,sales_growth_rate
2023-W21,328,-0.003,NaN,NaN,Stable,NaN,2751.0,Falling,1359.46,194.0,Falling,1067.07,-9.0,Stable,1566.10,2752.0,Rising,724.50,NaN,Rising,634.56,1110.0,NaN,673.70,2869.0,NaN,33.21,2292.0,Stable,466.76,2434.0,NaN,626.28,1913.0,NaN,1036.96,2266.0,NaN,1732.17,2220.0,Falling,200.84,1119.0,Stable,NaN,2610.0,NaN,NaN,NaN
2023-W22,280,0.088,NaN,NaN,Falling,16.14,2201.0,Stable,NaN,2119.0,Stable,1422.27,1590.0,NaN,541.13,NaN,Stable,1934.58,2870.0,Falling,865.59,1660.0,NaN,144.19,263.0,Rising,NaN,909.0,Stable,1234.13,120.0,Stable,784.18,1333.0,Falling,1972.79,2042.0,NaN,190.41,2746.0,Stable,NaN,789.0,Rising,348.40,NaN,Rising,1418.63,NaN
2023-W23,130,0.073,1231.77,1381.0,NaN,1656.94,NaN,Rising,262.30,76.0,Stable,135.74,2855.0,Rising,NaN,781.0,NaN,836.19,NaN,Rising,NaN,2469.0,Rising,139.51,384.0,Falling,1485.99,305.0,NaN,1827.78,2918.0,Stable,505.16,NaN,Rising,1490.95,50.0,Falling,840.78,864.0,NaN,537.78,NaN,Stable,440.98,1897.0,Stable,NaN,NaN
2023-W24,291,0.077,184.26,NaN,NaN,902.57,982.0,Rising,NaN,2139.0,Rising,1734.52,892.0,Rising,522.18,2568.0,Falling,1812.57,1033.0,NaN,392.05,2048.0,NaN,1934.38,1424.0,Rising,322.57,1959.0,Falling,1545.16,NaN,Stable,5.64,834.0,Stable,888.36,2207.0,Stable,NaN,605.0,Rising,NaN,178.0,Stable,NaN,614.0,Stable,217.92,NaN
2023-W25,200,-0.003,NaN,243.0,Rising,514.97,329.0,Stable,374.81,1232.0,Stable,84.69,2785.0,Rising,1297.35,NaN,Falling,NaN,1016.0,Falling,881.39,NaN,Stable,657.38,1328.0,NaN,871.83,1483.0,Falling,NaN,6.0,Stable,1533.28,355.0,Stable,NaN,2914.0,NaN,828.67,123.0,Stable,NaN,1348.0,Rising,434.84,NaN,Falling,1800.13,NaN


### week

In [22]:
# normalize
trend_report['week'] = trend_report['week'].astype(str).str.strip()

# convert to datetime (week start)
trend_report['week_start'] = pd.to_datetime(
    trend_report['week'] + '-1',
    format='%Y-W%U-%w',
    errors='coerce'
)

/var/folders/fb/r3xfzh555slbpdxcyfkj672w0000gn/T/ipykernel_11875/2031028760.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  trend_report['week'] = trend_report['week'].astype(str).str.strip()


In [23]:
trend_report.head()

,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,week,avg_users,sales_growth_rate,week_start
2023-W21,328,-0.003,NaN,NaN,Stable,NaN,2751.0,Falling,1359.46,194.0,Falling,1067.07,-9.0,Stable,1566.10,2752.0,Rising,724.50,NaN,Rising,634.56,1110.0,NaN,673.70,2869.0,NaN,33.21,2292.0,Stable,466.76,2434.0,NaN,626.28,1913.0,NaN,1036.96,2266.0,NaN,1732.17,2220.0,Falling,200.84,1119.0,Stable,NaN,2610.0,nan,NaN,NaN,NaT
2023-W22,280,0.088,NaN,NaN,Falling,16.14,2201.0,Stable,NaN,2119.0,Stable,1422.27,1590.0,NaN,541.13,NaN,Stable,1934.58,2870.0,Falling,865.59,1660.0,NaN,144.19,263.0,Rising,NaN,909.0,Stable,1234.13,120.0,Stable,784.18,1333.0,Falling,1972.79,2042.0,NaN,190.41,2746.0,Stable,NaN,789.0,Rising,348.40,NaN,Rising,1418.63,NaN,NaT
2023-W23,130,0.073,1231.77,1381.0,NaN,1656.94,NaN,Rising,262.30,76.0,Stable,135.74,2855.0,Rising,NaN,781.0,NaN,836.19,NaN,Rising,NaN,2469.0,Rising,139.51,384.0,Falling,1485.99,305.0,NaN,1827.78,2918.0,Stable,505.16,NaN,Rising,1490.95,50.0,Falling,840.78,864.0,NaN,537.78,NaN,Stable,440.98,1897.0,Stable,NaN,NaN,NaT
2023-W24,291,0.077,184.26,NaN,NaN,902.57,982.0,Rising,NaN,2139.0,Rising,1734.52,892.0,Rising,522.18,2568.0,Falling,1812.57,1033.0,NaN,392.05,2048.0,NaN,1934.38,1424.0,Rising,322.57,1959.0,Falling,1545.16,NaN,Stable,5.64,834.0,Stable,888.36,2207.0,Stable,NaN,605.0,Rising,NaN,178.0,Stable,NaN,614.0,Stable,217.92,NaN,NaT
2023-W25,200,-0.003,NaN,243.0,Rising,514.97,329.0,Stable,374.81,1232.0,Stable,84.69,2785.0,Rising,1297.35,NaN,Falling,NaN,1016.0,Falling,881.39,NaN,Stable,657.38,1328.0,NaN,871.83,1483.0,Falling,NaN,6.0,Stable,1533.28,355.0,Stable,NaN,2914.0,NaN,828.67,123.0,Stable,NaN,1348.0,Rising,434.84,NaN,Falling,1800.13,NaN,NaT


### Avg users

In [24]:
trend_report['avg_users'] = pd.to_numeric(trend_report['avg_users'], errors='coerce')
trend_report.loc[trend_report['avg_users'] < 0, 'avg_users'] = np.nan

### Sales Growth Rate'

In [25]:
trend_report['sales_growth_rate'] = pd.to_numeric(trend_report['sales_growth_rate'], errors='coerce')

In [26]:

#export verything
event_logs = event_logs.loc[:, :'amount']

event_logs.to_csv('event_logs(1).csv', index=False)
marketing_summary.to_csv('marketing_summary(1).csv', index=False)
trend_report.to_csv('trend_report(1).csv', index=False)

In [27]:
for name, df in [('event_logs', event_logs), 
                  ('marketing_summary', marketing_summary), 
                  ('trend_report', trend_report)]:
    print(f"\n{name}:")
    print((df.isnull().mean() * 100).round(1).to_string())


event_logs:
user_id         0.0
event_type      0.0
event_time    100.0
product_id    100.0
amount          0.0

marketing_summary:
date                100.0
users_active         34.0
total_sales          34.0
new_customers       100.0
report_generated    100.0

trend_report:
week                   0.0
avg_users             40.0
sales_growth_rate    100.0
week_start           100.0
